In [1]:
from pathlib import Path
import os
import sys
import json

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)

Project root: /home/hegde/code/terrainOS-demo


In [2]:
from eintelligence.data_prep.aoi import square_aoi
from eintelligence.data_prep.temporal_pairing import build_landcover_multisensor_manifest
from eintelligence.data_prep.manifest_utils import merge_record_manifests
from orchestrator.workflow_manager_landcover import (
    LandCoverWorkflowMS,
    TilingConfigLandCover,
    TrainingConfigLandCover,
)


In [3]:
PIPELINE_STAGE = "ingest"
# "ingest", "regional_only", "pooled_only", "infer_only"

CASE_NAME = "landcover"
SENSOR_MODE = "s1s2"

tiling_cfg = TilingConfigLandCover(
    bands_s2=("B02", "B03", "B04", "B08"),
    bands_s1=("vv", "vh"),
    tile_size=256,
    stride=256,
    max_cloud=50,
    sensor_mode=SENSOR_MODE,
)

train_cfg = TrainingConfigLandCover(
    batch_size=4,
    num_epochs=100,
    lr=1e-4,
    amp=False,   # keep false for now since you just debugged NaNs
)

wf = LandCoverWorkflowMS(project_root, tiling_cfg, train_cfg)


/home/hegde/code/terrainOS-demo/orchestrator/workflow_manager_landcover.py:206: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=(device.type == "cuda" and cfg.amp))


In [4]:
# %%
# Define multiple AOIs / date windows.
# Keep them sequential so downloads and tiling happen one run at a time.

RUN_SPECS = [
    {
        "region_name": "munich_lc_2023_summer",
        "aoi_geojson": square_aoi(48.1351, 11.5820),
        "start": "2023-06-01",
        "end": "2023-08-01",
        "aoi_id": "munich_core",
        "job_id": "munich_lc_2023_summer",
    },
    # Example:
    {
        "region_name": "novo_progresso_lc_2023_dry",
        "aoi_geojson": square_aoi(-7.754, -55.513),
        "start": "2023-06-01",
        "end": "2023-09-01",
        "aoi_id": "novo_progresso",
        "job_id": "novo_progresso_lc_2023_dry",
    },
]


In [5]:
corpus_dir = Path(project_root) / "data" / "corpus"
corpus_dir.mkdir(parents=True, exist_ok=True)

pooled_manifest = corpus_dir / "landcover_manifest_multisensor.json"
splits_path = corpus_dir / "landcover_splits.json"

models_dir = Path(project_root) / "models"
models_dir.mkdir(parents=True, exist_ok=True)

ckpt_path = models_dir / f"{CASE_NAME}_{SENSOR_MODE}.pt"
out_dir = corpus_dir / f"pred_{CASE_NAME}_{SENSOR_MODE}"

In [6]:
def regional_manifest_path(project_root: str, region_name: str) -> Path:
    return Path(project_root) / "data" / region_name / "S2" / "landcover_manifest_multisensor.json"

In [7]:
regional_manifest_paths = []
regional_namespaces = []

if PIPELINE_STAGE == "ingest":
    for spec in RUN_SPECS:
        region_name = spec["region_name"]
        print(f"\n=== INGEST: {region_name} ===")

        s1_coll, s2_coll, registry_path = wf.ingest_region(
            aoi_geojson=spec["aoi_geojson"],
            start=spec["start"],
            end=spec["end"],
            region_name=region_name,
            aoi_id=spec.get("aoi_id"),
            job_id=spec.get("job_id"),
        )

        landcover_manifest = build_landcover_multisensor_manifest(
            s2_collection_manifest_path=s2_coll,
            s1_collection_manifest_path=s1_coll,
            iou_min=0.8,
            worldcover_version="v200",
            worldcover_year="2021",
        )

        regional_manifest_paths.append(Path(landcover_manifest))
        regional_namespaces.append(region_name)
        print(f"[OK] built regional manifest: {landcover_manifest}")

elif PIPELINE_STAGE == "regional_only":
    for spec in RUN_SPECS:
        region_name = spec["region_name"]
        manifest_path = regional_manifest_path(project_root, region_name)

        if not manifest_path.is_file():
            raise RuntimeError(
                f"Expected regional manifest not found for {region_name}: {manifest_path}"
            )

        regional_manifest_paths.append(manifest_path)
        regional_namespaces.append(region_name)
        print(f"[OK] using existing regional manifest: {manifest_path}")

elif PIPELINE_STAGE in ("pooled_only", "infer_only"):
    print(f"[OK] skipping regional stage: {PIPELINE_STAGE}")

else:
    raise ValueError(f"Unsupported PIPELINE_STAGE: {PIPELINE_STAGE}")


=== INGEST: munich_lc_2023_summer ===


/home/hegde/code/terrainOS-demo/eintelligence/data_prep/tiler_streaming.py:368: RuntimeWarning: overflow encountered in power
  arr_stack = 10 ** (arr_stack / 10.0)  # dB to linear σ0


[INFO] Landcover multisensor manifest written to /home/hegde/code/terrainOS-demo/data/munich_lc_2023_summer/S2/landcover_manifest_multisensor.json with 76 tiles.
[OK] built regional manifest: /home/hegde/code/terrainOS-demo/data/munich_lc_2023_summer/S2/landcover_manifest_multisensor.json

=== INGEST: novo_progresso_lc_2023_dry ===
[INFO] Landcover multisensor manifest written to /home/hegde/code/terrainOS-demo/data/novo_progresso_lc_2023_dry/S2/landcover_manifest_multisensor.json with 288 tiles.
[OK] built regional manifest: /home/hegde/code/terrainOS-demo/data/novo_progresso_lc_2023_dry/S2/landcover_manifest_multisensor.json


In [8]:
# Step 2: merge all regional manifests into one pooled manifest

if PIPELINE_STAGE in ("ingest", "regional_only"):
    pooled_manifest = merge_record_manifests(
        manifest_paths=regional_manifest_paths,
        out_path=pooled_manifest,
        record_key="tiles",
        task_name="landcover_multisensor",
        namespaces=regional_namespaces,
        fields_to_prefix=("group_id", "tile_id", "scene_id"),
        set_default_aoi_id=True,
        deduplicate_on="tile_id",
        sort_by=("aoi_id", "group_id", "scene_id", "datetime", "row", "col", "tile_id"),
    )
    print(f"[OK] pooled manifest: {pooled_manifest}")

elif PIPELINE_STAGE in ("pooled_only", "infer_only"):
    if not pooled_manifest.is_file():
        raise RuntimeError(f"Pooled manifest not found: {pooled_manifest}")
    print(f"[OK] using existing pooled manifest: {pooled_manifest}")

[OK] pooled manifest: /home/hegde/code/terrainOS-demo/data/corpus/landcover_manifest_multisensor.json


In [9]:
if PIPELINE_STAGE == "infer_only":
    if not ckpt_path.is_file():
        raise RuntimeError(f"Checkpoint not found for infer_only: {ckpt_path}")

    wf.run(
        landcover_manifest=pooled_manifest,
        splits_path=splits_path,
        ckpt_path=ckpt_path,
        out_dir=out_dir,
        mode="infer",
        retrain=False,
        infer_split_name="test",
        stitch_scenes=False,
        max_tiles=None,
    )

else:
    wf.run(
        landcover_manifest=pooled_manifest,
        splits_path=splits_path,
        ckpt_path=ckpt_path,
        out_dir=out_dir,
        mode="train_and_infer",
        retrain=True,
        infer_split_name="test",
        stitch_scenes=False,
        max_tiles=None,
    )


Training land-cover head (WorldCover-supervised)...


/home/hegde/code/terrainOS-demo/orchestrator/workflow_manager_landcover.py:230: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.set_grad_enabled(train), autocast(enabled=(self.device.type == "cuda" and self.cfg.amp)):
/home/hegde/code/terrainOS-demo/orchestrator/workflow_manager_landcover.py:309: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type == "cuda" and self.cfg.amp)):


[epoch 00] train_loss=1.2377  val_loss=1.3803  mIoU=0.201  macroF1=0.273  OA=0.525
  ↳ saved best land-cover model to /home/hegde/code/terrainOS-demo/models/landcover_s1s2.pt (mIoU=0.201)
[epoch 01] train_loss=0.7629  val_loss=0.7600  mIoU=0.274  macroF1=0.353  OA=0.689
  ↳ saved best land-cover model to /home/hegde/code/terrainOS-demo/models/landcover_s1s2.pt (mIoU=0.274)
[epoch 02] train_loss=0.6333  val_loss=0.6450  mIoU=0.248  macroF1=0.328  OA=0.709
[epoch 03] train_loss=0.5419  val_loss=0.9495  mIoU=0.208  macroF1=0.292  OA=0.571
[epoch 04] train_loss=0.4892  val_loss=0.5419  mIoU=0.271  macroF1=0.349  OA=0.744
[epoch 05] train_loss=0.4592  val_loss=0.4847  mIoU=0.315  macroF1=0.386  OA=0.787
  ↳ saved best land-cover model to /home/hegde/code/terrainOS-demo/models/landcover_s1s2.pt (mIoU=0.315)
[epoch 06] train_loss=0.4444  val_loss=0.6742  mIoU=0.272  macroF1=0.351  OA=0.686
[epoch 07] train_loss=0.4315  val_loss=0.5496  mIoU=0.260  macroF1=0.337  OA=0.745
[epoch 08] train_loss

/home/hegde/code/terrainOS-demo/orchestrator/workflow_manager_landcover.py:611: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type == "cuda")):


wrote 56 land-cover tiles for split='test' -> /home/hegde/code/terrainOS-demo/data/corpus/pred_landcover_s1s2/test


In [10]:
# pooled_data = json.loads(Path(pooled_manifest).read_text())
# print("Total pooled tiles:", len(pooled_data["tiles"]))


In [11]:
# Step 3: define model + split/output locations

# models_dir = Path(project_root) / "models"
# models_dir.mkdir(parents=True, exist_ok=True)

# ckpt_path = models_dir / f"{CASE_NAME}_{SENSOR_MODE}.pt"
# splits_path = corpus_dir / "landcover_splits.json"
# out_dir = corpus_dir / f"pred_{CASE_NAME}_{SENSOR_MODE}"

# print("Checkpoint:", ckpt_path)
# print("Splits:", splits_path)
# print("Outputs:", out_dir)

In [12]:
# # Step 4: train on pooled train split, validate on val split, infer on pooled test split

# wf.run(
#     landcover_manifest=pooled_manifest,
#     splits_path=splits_path,
#     ckpt_path=ckpt_path,
#     out_dir=out_dir,
#     mode="train_and_infer",      # "train", "infer", "train_and_infer"
#     retrain=True,
#     infer_split_name="test",
#     stitch_scenes=True,
#     max_tiles=None,              # set an int for quick debugging
# )

In [13]:
# # Optional: inspect stitched outputs

# stitched_dir = out_dir / "test_stitched"
# print("Stitched outputs:", stitched_dir)